# 04 Constraint-Driven Structured Outputs (DSPy, 2026)

## What This Lesson Is
Apply constraints and post-validators to keep outputs parseable and policy-compliant.

## Scientific Lens
- Concept: Constrained generation with post-hoc validation
- Measure: Constraint pass rate and parse success rate
- Validity Limit: Constraint-conformant outputs can still be low quality.


## How It Works
1. Define deterministic validator.
2. Test invalid and valid cases.
3. Run live DSPy generation through validator.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Constraint lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
allowed_priority = {"low", "medium", "high"}

candidate = {"action": "rotate_keys", "owner": "platform", "priority": "high"}
required = {"action", "owner", "priority"}

missing = required - set(candidate)
if missing:
    raise ValueError(f"Missing: {missing}")
if candidate["priority"] not in allowed_priority:
    raise ValueError("priority invalid")

print("validated:", candidate)
assert candidate["priority"] == "high"


In [ ]:
# Live Demo
import json
import os

try:
    import dspy
except Exception as exc:
    print(f"Skipping live constrained demo: dspy unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live constrained demo: OPENAI_API_KEY not set.")
    else:
        dspy.configure(lm=dspy.LM("openai/gpt-4.1-mini", api_key=api_key, temperature=0))
        predict = dspy.Predict("task -> json_output")
        out = predict(task="Return JSON with keys action, owner, priority and valid priority in [low, medium, high].")
        print(out.json_output)
        try:
            parsed = json.loads(out.json_output)
            print(parsed)
        except Exception as exc:
            print(f"parse failed: {exc}")


## Applied Labs
1. Add strict regex constraints for owner identifiers.
2. Implement one automatic repair attempt before failing hard.
3. Track parse success over 30 live outputs and report failure taxonomy.

## Validation Checklist
- Validator enforces required fields and allowable values.
- Invalid outputs fail clearly with actionable reason.
- Live generation path feeds through same validator logic.

## Further Reading
- [DSPy Structured Programs](https://dspy.ai/learn/programming/modules/)
- [JSON Schema Validation](https://json-schema.org/understanding-json-schema/)
- [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)
